In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import harmonypy
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import figstyle as fs

ad.settings.zarr_write_format = 2

BASE = Path.cwd()
OUT_DIR = BASE / "results" / "00_preprocess_and_cluster"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RAW = BASE.parent / "xenium_data_mouse" / "integrated" / "integrated_mouse.zarr" / "tables" / "table"

In [ ]:
adata = ad.read_zarr(RAW)
adata

In [ ]:
SLIDES = ["No1", "No2"]
CONDITION_ORDER = ["C-ion_24_h", "C-ion_48_h", "C-ion_3.5_d", "C-ion_Sham", "gamma-ray_Sham", "gamma-ray_24_h", "gamma-ray_48_h", "gamma-ray_3.5_d"]
sort_key = pd.DataFrame({"slide": pd.Categorical(adata.obs["slide"].astype(str), SLIDES, ordered=True),
                      "roi": pd.Categorical(adata.obs["condition"].astype(str), CONDITION_ORDER, ordered=True),
                       "cell_id": adata.obs["cell_id"].astype(str)}, index=adata.obs_names)
adata = adata[sort_key.sort_values(["slide", "roi", "cell_id"]).index].copy()
adata.obs["roi_id"] = adata.obs["roi_id"].astype(str)
adata.obs["slide"] = adata.obs["slide"].astype(str)
adata.obs["condition"] = adata.obs["condition"].astype(str)
pd.crosstab(adata.obs["condition"], adata.obs["slide"])

In [ ]:
ROI_NO = {
    "C-ion_Sham_No1": 1,
    "C-ion_24_h_No1": 2,
    "C-ion_24_h_No2": 3,
    "C-ion_48_h_No1": 4,
    "C-ion_48_h_No2": 5,
    "C-ion_3.5_d_No1": 6,
    "C-ion_3.5_d_No2": 7,
    "gamma-ray_Sham_No1": 8,
    "gamma-ray_24_h_No1": 9,
    "gamma-ray_24_h_No2": 10,
    "gamma-ray_48_h_No1": 11,
    "gamma-ray_48_h_No2": 12,
    "gamma-ray_3.5_d_No1": 13,
    "gamma-ray_3.5_d_No2": 14,
}
RADIATION = {
    "C-ion_Sham": "C-ion",
    "C-ion_24_h": "C-ion",
    "C-ion_48_h": "C-ion",
    "C-ion_3.5_d": "C-ion",
    "gamma-ray_Sham": "γ-ray",
    "gamma-ray_24_h": "γ-ray",
    "gamma-ray_48_h": "γ-ray",
    "gamma-ray_3.5_d": "γ-ray",
}
TIME = {
    "C-ion_Sham": "Sham",
    "C-ion_24_h": "24 h",
    "C-ion_48_h": "48 h",
    "C-ion_3.5_d": "3.5 d",
    "gamma-ray_Sham": "Sham",
    "gamma-ray_24_h": "24 h",
    "gamma-ray_48_h": "48 h",
    "gamma-ray_3.5_d": "3.5 d",
}
SECTION = {
    1: "C-ion Sham (ROI 1)",
    2: "C-ion 24 h (ROI 2)",
    3: "C-ion 24 h (ROI 3)",
    4: "C-ion 48 h (ROI 4)",
    5: "C-ion 48 h (ROI 5)",
    6: "C-ion 3.5 d (ROI 6)",
    7: "C-ion 3.5 d (ROI 7)",
    8: "γ-ray Sham (ROI 8)",
    9: "γ-ray 24 h (ROI 9)",
    10: "γ-ray 24 h (ROI 10)",
    11: "γ-ray 48 h (ROI 11)",
    12: "γ-ray 48 h (ROI 12)",
    13: "γ-ray 3.5 d (ROI 13)",
    14: "γ-ray 3.5 d (ROI 14)",
}

adata.obs["roi_no"] = (adata.obs["condition"] + "_" + adata.obs["slide"]).map(ROI_NO)
adata.obs["radiation"] = adata.obs["condition"].map(RADIATION)
adata.obs["time"] = adata.obs["condition"].map(TIME)
adata.obs["section"] = adata.obs["roi_no"].map(SECTION)
adata.obs["roi_no"] = adata.obs["roi_no"].astype("category")
adata.obs["section"] = pd.Categorical(adata.obs["section"], categories=SECTION.values())

ROIS = (adata.obs[["roi_no", "condition", "radiation", "time", "slide", "section"]]
        .drop_duplicates().sort_values("roi_no").set_index("roi_no"))
ROIS.to_csv(OUT_DIR / "roi_numbers.csv")
ROIS

In [ ]:
MIN_COUNTS = 10
MIN_GENES = 5
MIN_CELLS = 3

sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
obs_before = adata.obs[["condition", "slide", "roi_no", "total_counts", "n_genes_by_counts"]].copy()
obs_before.to_parquet(OUT_DIR / "qc_per_cell_before.parquet")
n_genes_before = adata.n_vars
keep = (adata.obs["total_counts"] >= MIN_COUNTS) & (adata.obs["n_genes_by_counts"] >= MIN_GENES)
adata = adata[keep].copy()
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)
pd.crosstab(adata.obs["condition"], adata.obs["slide"])

In [ ]:
obs_after = adata.obs
rows = []
for condition in ROIS["condition"].unique():
    for slide in SLIDES:
        cells_before = obs_before[(obs_before["condition"] == condition) & (obs_before["slide"] == slide)]
        cells_after = obs_after[(obs_after["condition"] == condition) & (obs_after["slide"] == slide)]
        if len(cells_before) == 0:
            continue
        rows.append({
            "condition": condition,
            "slide": slide,
            "cells_before": len(cells_before),
            "cells_after": len(cells_after),
            "removed_pct": round((1 - len(cells_after) / len(cells_before)) * 100, 1),
            "median_transcripts_before": cells_before["total_counts"].median(),
            "median_transcripts_after": cells_after["total_counts"].median(),
            "median_genes_before": cells_before["n_genes_by_counts"].median(),
            "median_genes_after": cells_after["n_genes_by_counts"].median(),
        })

rows.append({
    "condition": "All",
    "slide": "All",
    "cells_before": len(obs_before),
    "cells_after": len(obs_after),
    "removed_pct": round((1 - len(obs_after) / len(obs_before)) * 100, 1),
    "median_transcripts_before": obs_before["total_counts"].median(),
    "median_transcripts_after": obs_after["total_counts"].median(),
    "median_genes_before": obs_before["n_genes_by_counts"].median(),
    "median_genes_after": obs_after["n_genes_by_counts"].median(),
})
qc = pd.DataFrame(rows).set_index(["condition", "slide"])
qc.to_csv(OUT_DIR / "qc_summary.csv")
total = pd.DataFrame({"before": [len(obs_before), n_genes_before], "after": [adata.n_obs, adata.n_vars]},
                     index=["cells", "genes"])
total.to_csv(OUT_DIR / "qc_total.csv")
display(total)
qc

In [ ]:
def qc_violin(ax, values, labels, cutoff, ylabel, colors):
    data = [np.log10(np.asarray(v, dtype=float) + 1) for v in values]
    xs = np.arange(1, len(data) + 1)
    parts = ax.violinplot(data, positions=xs, widths=0.8, showextrema=False)
    for body, color in zip(parts["bodies"], colors):
        body.set(facecolor=color, edgecolor=fs.BLACK, linewidth=0.3, alpha=1)
    ax.hlines([np.median(d) for d in data], xs - 0.25, xs + 0.25, color=fs.BLACK, lw=0.75)
    for c in cutoff:
        if c is not None:
            ax.axhline(np.log10(c + 1), color=fs.GREY, lw=0.6, ls=(0, (3, 2)))
    top = max(d.max() for d in data)
    ticks = [t for t in [0, 1, 10, 100, 1000, 10000, 100000] if np.log10(t + 1) <= top * 1.02]
    ax.set_yticks(np.log10(np.array(ticks) + 1), [f"{t:,}" for t in ticks])
    ax.set_xticks(xs, labels)
    ax.set_xlim(0.4, len(data) + 0.6)
    ax.set_ylabel(ylabel)

fig, (ax_counts, ax_genes) = plt.subplots(2, 1, figsize=(fs.COL2 / 2, 60 * fs.MM), sharex=True, layout="constrained")
fig.get_layout_engine().set(h_pad=0.5 * fs.MM, hspace=0)
labels = list(ROIS["section"])
colors = [fs.ROI_COLORS[c] for c in ROIS["condition"]]

counts, genes = [], []
for roi in ROIS.index:
    cells = obs_before[obs_before["roi_no"] == roi]
    counts.append(cells["total_counts"])
    genes.append(cells["n_genes_by_counts"])

qc_violin(ax_counts, counts, labels, (MIN_COUNTS,), "Transcripts per cell", colors=colors)
qc_violin(ax_genes, genes, labels, (MIN_GENES,), "Genes per cell", colors=colors)
plt.setp(ax_genes.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
fig.align_ylabels([ax_counts, ax_genes])
fs.save(fig, OUT_DIR / "qc_violin_by_roi")
plt.show()

In [ ]:
with threadpool_limits(limits=1):
    adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=100)
    sc.pp.log1p(adata)
    adata.layers["log1p_norm"] = adata.X.copy()
    adata.raw = adata
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, n_comps=50, svd_solver="arpack", random_state=42)

In [ ]:
with threadpool_limits(limits=1):
    harmony = harmonypy.run_harmony(np.ascontiguousarray(adata.obsm["X_pca"], dtype=np.float64), adata.obs[["slide"]].astype(str),
                               ["slide"], max_iter_harmony=20, random_state=42, device="cpu")
adata.obsm["X_pca_harmony"] = np.ascontiguousarray(harmony.Z_corr, dtype=np.float32)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50, use_rep="X_pca_harmony", metric="cosine", random_state=42)
sc.tl.leiden(adata, resolution=0.2, key_added='leiden_rough')
sc.tl.umap(adata, min_dist=0.3, spread=1.0, random_state=42)
ax = sc.pl.umap(adata, color="leiden_rough", legend_loc="on data", legend_fontsize=6, legend_fontweight="normal",
                frameon=False, title="", size=0.2, show=False)
fig = ax.figure
fig.set_size_inches(fs.COL1, fs.COL1)
fs.save(fig, OUT_DIR / "umap_leiden_rough")
plt.show()
adata.obs["leiden_rough"].value_counts().sort_index()

In [ ]:
ax = sc.pl.umap(adata, color="slide", frameon=False, title="", size=0.2, show=False)
fig = ax.figure
fig.set_size_inches(fs.COL1, fs.COL1)
fs.save(fig, OUT_DIR / "umap_slide")
plt.show()

In [ ]:
# adata.write_zarr(BASE / "data" / "integrated_mouse_preprocessed.zarr")